<a href="https://colab.research.google.com/github/Naveen-gale/deep_learning/blob/main/rnn_text_genration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense
from tensorflow.keras.utils import to_categorical

In [3]:
from google.colab import files

uploaded = files.upload()

Saving tiny-shakespeare.txt to tiny-shakespeare.txt


In [1]:
with open('tiny-shakespeare.txt', 'r') as f:
    data = f.read()

In [2]:
print("Characters:", len(data))
print("Words:", len(data.split()))

Characters: 1115394
Words: 202651


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

In [4]:
tokenizer.fit_on_texts(data)

In [5]:
total_words = len(tokenizer.word_index) + 1

print(total_words)

12652


In [6]:
input_sequences = []

for line in data.split("\n"):

    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

In [7]:
print(input_sequences[:10])

[[112, 292], [162, 57], [162, 57, 992], [162, 57, 992, 166], [162, 57, 992, 166, 692], [162, 57, 992, 166, 692, 151], [162, 57, 992, 166, 692, 151, 36], [162, 57, 992, 166, 692, 151, 36, 126], [126, 126], [112, 292]]


In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import numpy as np

max_len = max(len(seq) for seq in input_sequences)

input_sequences = np.array(
    pad_sequences(
        input_sequences,
        maxlen=max_len,
        padding="pre"
    )
)

In [14]:
X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = to_categorical(y, num_classes=total_words)

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Shape of X: (171312, 15)
Shape of y: (171312, 12652)


Now, let's define the LSTM model. It will consist of:
1.  An `Embedding` layer to convert word indices into dense vectors.
2.  An `LSTM` layer to capture sequential dependencies.
3.  A `Dense` output layer with a `softmax` activation to predict the next word.

In [2]:
model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_len-1))
model.add(LSTM(150))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

NameError: name 'Sequential' is not defined

Now that the model is defined and compiled, let's train it. Training a language model on a large dataset like Tiny Shakespeare can take some time. I will set the number of epochs to a reasonable value (e.g., 50) and use a batch size that fits in memory.

In [1]:
history = model.fit(X, y, epochs=50, verbose=1)

NameError: name 'model' is not defined

Now that the model is trained, we can use it to generate text. First, let's create a helper function to generate text given a seed phrase.

In [ ]:
def generate_text(seed_text, next_words, model, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)
        predicted = np.argmax(predicted_probs, axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

Let's try generating some text with a sample seed phrase.

In [ ]:
seed_text = "first citizen"
generated_words = 20
max_sequence_len = max_len # Use the previously calculated max_len

print(generate_text(seed_text, generated_words, model, max_sequence_len))